# Regularisation

In notebook 01 you saw that a flexible model can fit the training data very well and still fail on new data. The plain [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model has no setting to stop this: it always uses every feature with whatever coefficient fits the training data best.

**Regularisation** adds that setting. It changes what the model minimises: besides the error, the model also pays a penalty for large coefficients. This notebook compares three regularised linear models from scikit-learn, **Ridge**, **Lasso** and **ElasticNet**, with plain linear regression.

## Learning Objectives

At the end of this notebook, you should be able to:

- Standardise the features before fitting a regularised model.
- Train Ridge and Lasso models and choose their regularisation strength `alpha` on a validation set.
- Compare how `alpha` changes the size and the number of the coefficients.
- Train an ElasticNet model that combines the Ridge and Lasso penalties.

## The Data

We use the **Wine Quality Dataset**. It has 11 physicochemical measurements of white wines, and the target is the wine's **quality**, a score between 0 and 10 given by tasters.

| Features | |
|---|---|
| fixed acidity, volatile acidity, citric acid | acidity |
| residual sugar, alcohol, density | sugar and alcohol content |
| chlorides, sulphates, pH | salts and acidity level |
| free sulfur dioxide, total sulfur dioxide | preservatives |

To give the models something to overfit, we will also expand the dataset with **interaction features**: all products of two features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn import set_config

# Make every scikit-learn transformer return a pandas DataFrame instead of a NumPy array
set_config(transform_output="pandas")

# Define random seed
RSEED = 0

# Define figure size
plt.rcParams["figure.figsize"] = (10, 7)

In [ ]:
df = pd.read_csv("data/winequality-white.csv", sep=";")
df.head(2)

In [ ]:
df.info()

All features are numeric and there are no missing values, so we can skip data cleaning and focus on the models.

In [ ]:
# Define features and target
X = df.drop(columns="quality")
y = df["quality"]

In [ ]:
# Correlation plot
ax = sns.heatmap(df.corr(), annot=True, fmt=".2f")

## Feature Engineering

We create **interaction terms**: the products of every pair of features, for example `alcohol * density`. With three features $X_1, X_2, X_3$ this gives six features: $X_1, X_2, X_3, X_1 X_2, X_1 X_3, X_2 X_3$.

The [documentation of `PolynomialFeatures`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) lists all the options.

In [ ]:
# Generate interaction features
poly = PolynomialFeatures(interaction_only=True, include_bias=False, degree=2)
X_poly = poly.fit_transform(X)

The number of pairs you can pick from $n$ features is the number of [combinations](https://en.wikipedia.org/wiki/Combination) of 2 out of $n$:

$$\binom{n}{2} = \frac{n(n-1)}{2}$$

With 11 features that is 55 pairs, plus the 11 original features:

In [ ]:
# 11 features + 55 pairs = 66 features
poly.n_output_features_

## Train-Validation-Test Split

As in the diabetes project, we split the data three ways:

- the **training set** to fit the models,
- the **validation set** to compare versions and choose `alpha`,
- the **test set**, kept aside until the very end for one final score.

The whole workflow of this notebook, from the split to the final score:

<img src="assets/validation_workflow.png" width="500">

Documentation: [`train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).

In [ ]:
# First split: set the test set aside
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=RSEED
)

# Second split: separate the validation set from the training set
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=RSEED
)

print("Training samples:  ", X_train.shape[0], "  Features:", X_train.shape[1])
print("Validation samples:", X_val.shape[0])
print("Test samples:      ", X_test.shape[0])

## Scaling the Features

Ridge, Lasso and ElasticNet penalise the size of every coefficient in the same way. But the size of a coefficient depends on the units of its feature. `density` barely varies (standard deviation 0.003), while `total sulfur dioxide` varies a lot (standard deviation 42). To have the same effect, `density` needs a huge coefficient and `total sulfur dioxide` a tiny one, so without scaling the penalty would punish features for their units, not for their importance.

The fix is to **standardise** every feature: subtract its mean and divide by its standard deviation, so that every feature has mean 0 and standard deviation 1.

> [!IMPORTANT]
> Fit the scaler on the **training set only**, then apply it to the other sets. The validation and test sets must stay unseen, exactly as with the split itself.

Scaling comes back tomorrow in W3D3 (Feature Engineering), where you will scale numeric features as one step of a preprocessing pipeline.

Documentation: [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_train_scaled.iloc[:, :3].describe().loc[["mean", "std"]].round(2)

## Linear Regression

We start with plain linear regression on all 66 features. To compare the models fairly, the function below fits a model on the training set and reports the **RMSE** (root mean squared error) on the training and the validation set, how many features the model uses, and its largest coefficient.

Documentation: [`root_mean_squared_error`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html).

In [ ]:
def evaluate(model, name):
    """Fit the model on the scaled training data and print how it does."""
    model.fit(X_train_scaled, y_train)
    rmse_train = root_mean_squared_error(y_train, model.predict(X_train_scaled))
    rmse_val = root_mean_squared_error(y_val, model.predict(X_val_scaled))
    coefficients = pd.Series(model.coef_, index=X_train_scaled.columns)
    n_used = (coefficients != 0).sum()
    largest = coefficients.abs().max()
    print(
        f"{name:20} train RMSE {rmse_train:.3f} | validation RMSE {rmse_val:.3f} | "
        f"{n_used:2} of {len(model.coef_)} features | largest coefficient {largest:.2f}"
    )
    return model


lin_reg = evaluate(LinearRegression(), "Linear regression")

The model fits the training data better (RMSE 0.693) than the validation data (0.736). Look at the largest coefficient: 120.76, on features that all have a standard deviation of 1. The 66 features are strongly correlated with each other (many are products of the same measurements), and the model pairs huge positive and negative coefficients that cancel out on the training data. That is overfitting in the coefficients, and it makes the model fragile on new data, as you will see on the test set at the end.

## Ridge Regression (L2 Regularisation)

**Ridge** regression minimises the usual squared error plus a penalty on the **squared** coefficients:

$$\text{error} + \alpha \sum_{j} b_j^2$$

The hyperparameter $\alpha$ (`alpha` in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)) sets how strong the penalty is. A larger `alpha` pushes the coefficients closer to zero: the model fits the training data a little worse, but may generalise better. With `alpha = 0` Ridge is plain linear regression.

Let's try four values:

In [ ]:
ridge_models = {}
for alpha in [1, 10, 100, 1000]:
    ridge_models[alpha] = evaluate(Ridge(alpha=alpha), f"Ridge alpha={alpha}")

The validation RMSE improves from 0.729 at `alpha = 1` to **0.721 at `alpha = 100`**, better than linear regression's 0.736. At `alpha = 1000` the penalty is too strong: the model **underfits**, and both the training and the validation RMSE get worse (0.732 and 0.746). So `alpha = 100` is the best Ridge model here.

Notice two more things:

- The largest coefficient shrinks from 120.76 (linear regression) to 1.01, 0.45, 0.26 and 0.09 as `alpha` grows.
- Ridge still uses all 66 features: it makes coefficients small, but never exactly zero.

This is why we chose `alpha` on the **validation set**: had we used the test set, it would no longer be unseen, and the final score would be too optimistic. From W3D4 you will choose hyperparameters with cross-validation, which reuses the data more efficiently.

Plotting the coefficients shows the shrinkage. Each point is one of the 66 coefficients; many linear regression coefficients fall outside the plotted range.

In [ ]:
plt.plot(lin_reg.coef_, "o", alpha=0.5, label="Linear regression", color="green")
plt.plot(ridge_models[1].coef_, "v", label="Ridge alpha=1", color="orange")
plt.plot(ridge_models[10].coef_, "*", label="Ridge alpha=10", color="red")
plt.plot(ridge_models[100].coef_, "s", alpha=0.5, label="Ridge alpha=100", color="blue")

plt.xlabel("Coefficient index", fontsize=16)
plt.ylabel("Coefficient value", fontsize=16)
plt.hlines(0, 0, len(lin_reg.coef_))
plt.ylim(-2, 2)
plt.legend(fontsize=13, ncol=2, loc=(0, 1.05));

### Try It: Skip the Scaling

Fit `Ridge(alpha=100)` on the **unscaled** training data (`X_train` instead of `X_train_scaled`) and calculate its validation RMSE on `X_val`. How does it compare with the scaled model (0.721)? For a closer look, compare the coefficient of `density` in both models.

In [ ]:
# TODO: fit Ridge(alpha=100) on the unscaled training data, X_train
ridge_unscaled = ...

# TODO: calculate its validation RMSE on the unscaled X_val
rmse_val = ...
print(f"unscaled validation RMSE: {rmse_val:.3f}")

# Compare the coefficient of density in both models
print("density, unscaled:", pd.Series(ridge_unscaled.coef_, index=X_train.columns)["density"])
print("density, scaled:  ", pd.Series(ridge_models[100].coef_, index=X_train.columns)["density"])

<details><summary>Show answer</summary>

```python
ridge_unscaled = Ridge(alpha=100).fit(X_train, y_train)
print(root_mean_squared_error(y_val, ridge_unscaled.predict(X_val)))
print("density, unscaled:", pd.Series(ridge_unscaled.coef_, index=X_train.columns)["density"])
print("density, scaled:  ", pd.Series(ridge_models[100].coef_, index=X_train.columns)["density"])
```

Without scaling, the validation RMSE is **0.745**: worse than the scaled model (0.721), and even worse than plain linear regression (0.736).

The reason shows in `density`. After scaling it is the **second most important** of the 66 features (coefficient -0.214). Unscaled, it drops to **65th**. Because `density` barely varies (standard deviation 0.003), it needs a very large coefficient to have any effect, and the penalty punishes exactly that. The model loses one of its best features because of its units, not its importance.

</details>

## Lasso Regression (L1 Regularisation)

**Lasso** penalises the **absolute** values of the coefficients instead:

$$\text{error} + \alpha \sum_{j} |b_j|$$

This small change has a big effect: Lasso sets some coefficients **exactly to zero**, so the model ignores those features completely. Lasso therefore also selects features, which you will use again in notebook 03. A larger `alpha` again means a stronger penalty ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)).

In [ ]:
lasso_models = {}
for alpha in [0.001, 0.01, 0.1]:
    lasso_models[alpha] = evaluate(Lasso(alpha=alpha, max_iter=100_000), f"Lasso alpha={alpha}")

Lasso keeps **39, 13 and 5** of the 66 features as `alpha` grows. The validation RMSE is 0.730 with 39 features and 0.732 with only 13, almost the same. At `alpha = 0.1` the model underfits (0.777).

That is Lasso's appeal: `alpha = 0.01` gives nearly the accuracy of the best models with a fifth of the features, a model that is much easier to explain. If accuracy is all that matters, Ridge with `alpha = 100` is still slightly better here (0.721).

In [ ]:
plt.plot(lin_reg.coef_, "o", alpha=0.3, label="Linear regression", color="green")
plt.plot(lasso_models[0.001].coef_, "v", alpha=0.6, label="Lasso alpha=0.001", color="orange")
plt.plot(lasso_models[0.01].coef_, "s", alpha=0.5, label="Lasso alpha=0.01", color="blue")
plt.plot(lasso_models[0.1].coef_, "*", alpha=0.6, label="Lasso alpha=0.1", color="red")

plt.xlabel("Coefficient index", fontsize=16)
plt.ylabel("Coefficient value", fontsize=16)
plt.hlines(0, 0, len(lin_reg.coef_))
plt.ylim(-1, 1)
plt.legend(fontsize=13, ncol=2, loc=(0, 1.05));

Most Lasso coefficients sit exactly on the zero line, especially for `alpha = 0.1`. The Ridge coefficients in the previous plot were small, but none of them was exactly zero.

## ElasticNet: Both Penalties

**ElasticNet** combines the two penalties. Besides `alpha`, it has a second hyperparameter, `l1_ratio`, that sets the mix: `l1_ratio = 1` is Lasso, `l1_ratio = 0` is Ridge, and values in between blend them ([documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)).

The blend helps when features come in correlated groups, as our interaction terms do. Lasso tends to keep one feature of a group and drop the others, somewhat arbitrarily; ElasticNet tends to keep or drop correlated features together.

In [ ]:
elastic_net = evaluate(
    ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=100_000), "ElasticNet alpha=0.01"
)

With an even mix (`l1_ratio = 0.5`), ElasticNet reaches a validation RMSE of 0.724 with 27 features: close to the best Ridge model, while still dropping more than half of the features.

| Model | Validation RMSE | Features used |
|---|---|---|
| Linear regression | 0.736 | 66 |
| Ridge, `alpha = 100` | **0.721** | 66 |
| Lasso, `alpha = 0.01` | 0.732 | 13 |
| ElasticNet, `alpha = 0.01` | 0.724 | 27 |

### Try It: Tune the Mix

Train two more ElasticNet models with `alpha = 0.01`: one with `l1_ratio = 0.2` (closer to Ridge) and one with `l1_ratio = 0.8` (closer to Lasso). How many features does each keep, and how does its validation RMSE compare with the table above?

In [ ]:
for ratio in [0.2, 0.8]:
    # TODO: create an ElasticNet with alpha=0.01, l1_ratio=ratio and max_iter=100_000
    model = ...
    evaluate(model, f"ElasticNet l1_ratio={ratio}")

<details><summary>Show answer</summary>

```python
for ratio in [0.2, 0.8]:
    evaluate(ElasticNet(alpha=0.01, l1_ratio=ratio, max_iter=100_000), f"ElasticNet l1_ratio={ratio}")
```

| `l1_ratio` | Validation RMSE | Features used |
|---|---|---|
| 0.2 | 0.722 | 41 |
| 0.5 | 0.724 | 27 |
| 0.8 | 0.729 | 19 |

As `l1_ratio` moves towards Lasso, the model keeps fewer features and loses a little accuracy. At 0.2 it is almost as accurate as the best Ridge model (0.721) while still dropping 25 features. `l1_ratio` is a second dial between accuracy and simplicity, and like `alpha` you choose it on the validation set.

</details>

## Final Check on the Test Set

We choose **Ridge with `alpha = 100`**, the best model on the validation set. As in the diabetes project, we retrain it on the training and validation data together, and only then score it once on the test set. The scaler is refitted on the same data.

For this lesson only, we also score plain linear regression, to see what regularisation gains. In a real project you would score only the model you chose.

In [ ]:
# Refit the scaler on the training and validation data together
final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_scaled = final_scaler.transform(X_test)

for name, model in [("Ridge alpha=100", Ridge(alpha=100)), ("Linear regression", LinearRegression())]:
    model.fit(X_train_val_scaled, y_train_val)
    rmse_test = root_mean_squared_error(y_test, model.predict(X_test_scaled))
    print(f"{name:18} test RMSE {rmse_test:.3f}")

Ridge scores a test RMSE of **0.794**. Plain linear regression, which looked only a little worse on the validation set, scores **1.105**: its huge, cancelling coefficients break down on the new data. That is the real value of regularisation. It does not only improve the score a little; it makes the model **stable**, so it does not fall apart on data it has not seen.

## Try It: A Finer Search for Alpha

Train a Lasso model with `alpha = 0.005` using the `evaluate()` function. How many features does it keep, and how does its validation RMSE compare with `alpha = 0.001` and `alpha = 0.01`?

In [ ]:
# TODO: create a Lasso with alpha=0.005 and max_iter=100_000
model = ...
evaluate(model, "Lasso alpha=0.005")

<details><summary>Show answer</summary>

`evaluate(Lasso(alpha=0.005, max_iter=100_000), "Lasso alpha=0.005")` keeps 24 of the 66 features, with a validation RMSE of 0.724. That is better than both `alpha = 0.001` (0.730, 39 features) and `alpha = 0.01` (0.732, 13 features): the best `alpha` often lies between the values you try first, so it pays to search more finely.

</details>

## Summary

In this notebook you:

- Standardised the features, fitting the scaler on the training set only, so that the penalty treats every feature the same.
- Trained Ridge and Lasso models and chose `alpha` on the validation set: Ridge with `alpha = 100` was best (validation RMSE 0.721).
- Compared how `alpha` changes the coefficients: Ridge shrinks them all but keeps every feature, while Lasso sets many of them to exactly zero (13 of 66 features kept at `alpha = 0.01`).
- Trained an ElasticNet model that blends both penalties (validation RMSE 0.724 with 27 features).

On the test set, the chosen Ridge model scored 0.794, while plain linear regression broke down at 1.105. In practice, Ridge is a good first choice; Lasso or ElasticNet are worth it when you want a smaller model that is easier to explain.

## References & Further Reading

- [**Linear models (scikit-learn)**](https://scikit-learn.org/stable/modules/linear_model.html): the user guide for Ridge, Lasso and ElasticNet, with the formulas behind each penalty.
- [**StandardScaler (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html): the scaler used in this notebook.
- **Introduction to Machine Learning with Python** by Andreas C. Müller and Sarah Guido: the chapter on supervised learning covers linear models and regularisation with similar examples.